In [24]:
#NCSG Team 5 Data Dashboard 4/22/26

from pathlib import Path
import requests
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

FRED_API_KEY = "9e5d944924918a26660adf4a492e447e"

OUTPUT_DIR = Path("plotly_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

FRED_SERIES_TITLES = {
    "unemployment_rate": "Unemployment Rate in Maryland",
    "unemployed_persons": "Unemployed Persons in Maryland",
    "labor_force_participation_rate": "Labor Force Participation Rate for Maryland",
    "employment": "Employment Level for Maryland",
    "unemployment_claims": "Initial Claims in Maryland",
    "civilian_labor_force": "Civilian Labor Force in Maryland",
    "discouraged_workers": "Not in Labor Force: Discouraged Workers for Maryland",
    "average_hourly_earnings": "Average Hourly Earnings of All Employees: Total Private in Maryland"
}

In [23]:
# Given an exact series title, search FRED and return the series ID. If multiple matches are found, return the first one.
def fred_search_series_id(exact_title: str) -> str:
    url = "https://api.stlouisfed.org/fred/series/search"
    params = {
        "search_text": exact_title,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "limit": 10
    }

    res = requests.get(url, params=params, timeout=30)
    res.raise_for_status()
    payload = res.json()

    series_list = payload.get("seriess", [])
    if not series_list:
        raise ValueError(f"No FRED search results for: {exact_title}")

    for s in series_list:
        if s.get("title", "").strip().lower() == exact_title.strip().lower():
            return s["id"]

    return series_list[0]["id"]

# Given a FRED series ID, fetch the observations and return a cleaned DataFrame with 'date' and 'value' columns.
def fred_fetch_observations(series_id: str) -> pd.DataFrame:
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json"
    }

    res = requests.get(url, params=params, timeout=30)
    res.raise_for_status()
    payload = res.json()

    obs = payload.get("observations", [])
    if not obs:
        raise ValueError(f"No observations returned for {series_id}")

    df = pd.DataFrame(obs)
    df = df[df["value"] != "."].copy()
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df[["date", "value"]].dropna().sort_values("date")
    return df

In [22]:
labor_frames = []
resolved_series = {}

# For each metric, search FRED for the exact title, fetch the series ID, then fetch the observations and store in a DataFrame. Also keep track of resolved series IDs.
for metric_key, exact_title in FRED_SERIES_TITLES.items():
    series_id = fred_search_series_id(exact_title)
    resolved_series[metric_key] = series_id

    df = fred_fetch_observations(series_id)
    df["metric_key"] = metric_key
    df["metric"] = exact_title
    df["series_id"] = series_id
    labor_frames.append(df)

labor_df = pd.concat(labor_frames, ignore_index=True)

print("Resolved FRED series:")
for k, v in resolved_series.items():
    print(f"{k}: {v}")

labor_df.head()

Resolved FRED series:
unemployment_rate: MDUR
unemployed_persons: LAUST240000000000003A
labor_force_participation_rate: LBSSA24
employment: EMPLOYMD
unemployment_claims: MDICLAIMS
civilian_labor_force: MDLF
discouraged_workers: DISCWORKMD
average_hourly_earnings: SMU24000000500000003


,date,value,metric_key,metric,series_id
0,1976-01-01,6.5,unemployment_rate,Unemployment Rate in Maryland,MDUR
1,1976-02-01,6.5,unemployment_rate,Unemployment Rate in Maryland,MDUR
2,1976-03-01,6.5,unemployment_rate,Unemployment Rate in Maryland,MDUR
3,1976-04-01,6.5,unemployment_rate,Unemployment Rate in Maryland,MDUR
4,1976-05-01,6.6,unemployment_rate,Unemployment Rate in Maryland,MDUR


In [31]:
# ALL CHARTS IN ONE

titles_in_order = [
    FRED_SERIES_TITLES["unemployment_rate"],
    FRED_SERIES_TITLES["unemployed_persons"],
    FRED_SERIES_TITLES["labor_force_participation_rate"],
    FRED_SERIES_TITLES["employment"],
    FRED_SERIES_TITLES["unemployment_claims"],
    FRED_SERIES_TITLES["civilian_labor_force"],
    FRED_SERIES_TITLES["discouraged_workers"],
    FRED_SERIES_TITLES["average_hourly_earnings"]
]

fig = make_subplots(
    rows=len(titles_in_order),
    cols=1,
    subplot_titles=titles_in_order,
    vertical_spacing=0.05
)

for i, title in enumerate(titles_in_order, start=1):
    df_m = labor_df[labor_df["metric"] == title].sort_values("date")

    if title == FRED_SERIES_TITLES["unemployment_claims"]:
        trace = go.Bar(
            x=df_m["date"],
            y=df_m["value"],
            name=title,
            showlegend=False,
            marker=dict(
                color="crimson",  # 🔥 stronger color
                line=dict(color="black", width=0.3)
            ),
            width=1000 * 60 * 60 * 24 * 7  # 👈 makes weekly bars thicker
        )
        y_title = "Claims"

    elif title == FRED_SERIES_TITLES["average_hourly_earnings"]:
        trace = go.Scatter(
            x=df_m["date"],
            y=df_m["value"],
            mode="lines+markers",
            name=title,
            showlegend=False
        )
        y_title = "Dollars per Hour"

    elif title in [
        FRED_SERIES_TITLES["unemployment_rate"],
        FRED_SERIES_TITLES["labor_force_participation_rate"]
    ]:
        trace = go.Scatter(
            x=df_m["date"],
            y=df_m["value"],
            mode="lines+markers",
            name=title,
            showlegend=False
        )
        y_title = "Percent"

    else:
        trace = go.Scatter(
            x=df_m["date"],
            y=df_m["value"],
            mode="lines+markers",
            name=title,
            showlegend=False
        )
        y_title = "Persons"

    fig.add_trace(trace, row=i, col=1)
    fig.update_yaxes(title_text=y_title, row=i, col=1)

fig.update_layout(
    height=2500,
    title="Maryland Labor Dashboard",
    template="plotly_white"
)

fig.show()

In [8]:
title = FRED_SERIES_TITLES["unemployment_rate"]
df_m = labor_df[labor_df["metric"] == title].sort_values("date")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_m["date"],
    y=df_m["value"],
    mode="lines+markers",
    name="Unemployment Rate"
))

fig.update_layout(
    title="Unemployment Rate — Maryland",
    xaxis_title="Date",
    yaxis_title="Percent",
    template="plotly_white"
)

fig.show()

In [32]:
# For the "Unemployed Persons" metric, we'll use a line chart with markers, and set the y-axis title to "Persons".
title = FRED_SERIES_TITLES["unemployed_persons"]
df_m = labor_df[labor_df["metric"] == title].sort_values("date")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_m["date"],
    y=df_m["value"],
    mode="lines+markers",
    name="Unemployed Persons"
))

fig.update_layout(
    title="Unemployed Persons — Maryland",
    xaxis_title="Date",
    yaxis_title="Persons",
    template="plotly_white"
)

fig.show()

In [33]:
# For the "Labor Force Participation Rate" metric, we'll also use a line chart with markers, and set the y-axis title to "Percent".
title = FRED_SERIES_TITLES["labor_force_participation_rate"]
df_m = labor_df[labor_df["metric"] == title].sort_values("date")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_m["date"],
    y=df_m["value"],
    mode="lines+markers",
    name="Labor Force Participation Rate"
))

fig.update_layout(
    title="Labor Force Participation Rate — Maryland",
    xaxis_title="Date",
    yaxis_title="Percent",
    template="plotly_white"
)

fig.show()

In [11]:
title = FRED_SERIES_TITLES["employment"]
df_m = labor_df[labor_df["metric"] == title].sort_values("date")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_m["date"],
    y=df_m["value"],
    mode="lines+markers",
    name="Employment"
))

fig.update_layout(
    title="Employment Level — Maryland",
    xaxis_title="Date",
    yaxis_title="Persons",
    template="plotly_white"
)

fig.show()

In [34]:
# For the "Initial Unemployment Claims" metric, we'll use a bar chart to better visualize the weekly claims data, and set the y-axis title to "Claims".
title = FRED_SERIES_TITLES["unemployment_claims"]
df_m = labor_df[labor_df["metric"] == title].sort_values("date")

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_m["date"],
    y=df_m["value"],
    mode="lines",
    line=dict(color="steelblue", width=2)
))

fig.update_layout(
    title="Initial Unemployment Claims — Maryland",
    xaxis_title="Date",
    yaxis_title="Claims",
    template="plotly_white",
    xaxis=dict( #added a slider to be able to see all of the data without it being too cluttered
        range=[
            df_m["date"].max() - pd.DateOffset(years=5),
            df_m["date"].max()
        ],
        rangeslider=dict(visible=True)
    )
)

fig.show()

In [35]:
# For the "Civilian Labor Force" metric, we'll use a line chart with markers, and set the y-axis title to "Persons".
title = FRED_SERIES_TITLES["civilian_labor_force"]
df_m = labor_df[labor_df["metric"] == title].sort_values("date")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_m["date"],
    y=df_m["value"],
    mode="lines+markers",
    name="Civilian Labor Force"
))

fig.update_layout(
    title="Civilian Labor Force — Maryland",
    xaxis_title="Date",
    yaxis_title="Persons",
    template="plotly_white"
)

fig.show()

In [36]:
# For the "Discouraged Workers" metric, we'll use a line chart with markers, and set the y-axis title to "Persons".
title = FRED_SERIES_TITLES["discouraged_workers"]
df_m = labor_df[labor_df["metric"] == title].sort_values("date")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_m["date"],
    y=df_m["value"],
    mode="lines+markers",
    name="Discouraged Workers"
))

fig.update_layout(
    title="Discouraged Workers — Maryland",
    xaxis_title="Date",
    yaxis_title="Persons",
    template="plotly_white"
)

fig.show()

In [37]:
# For the "Average Hourly Earnings" metric, we'll use a line chart with markers, and set the y-axis title to "Dollars per Hour".
title = FRED_SERIES_TITLES["average_hourly_earnings"]
df_m = labor_df[labor_df["metric"] == title].sort_values("date")

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_m["date"],
    y=df_m["value"],
    mode="lines+markers",
    name="Average Hourly Earnings"
))

fig.update_layout(
    title="Average Hourly Earnings of All Employees: Total Private — Maryland",
    xaxis_title="Date",
    yaxis_title="Dollars per Hour",
    template="plotly_white"
)

fig.show()